In [14]:
"""
===============================================================================
PROJETO: Impacto de IA nos Estudantes (AI Student Impact Dataset)
ARQUIVO: EDA_baseline.py
OBJETIVO: Análise Exploratória de Dados (EDA) automática, comparação de baselines
          de regressão e inferência no conjunto de teste.
AUTOR: Senior Data Scientist Agent
DATA: 2026-09-17
===============================================================================
"""


'\n===============================================================================\nPROJETO: Impacto de IA nos Estudantes (AI Student Impact Dataset)\nARQUIVO: EDA_baseline.py\nOBJETIVO: Análise Exploratória de Dados (EDA) automática, comparação de baselines\n          de regressão e inferência no conjunto de teste.\nAUTOR: Senior Data Scientist Agent\nDATA: 2026-09-17\n===============================================================================\n'

# 1. Importação de Bibliotecas e Configurações


In [15]:
import os
import sys
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error

# Formatação de saída do pandas para melhor legibilidade
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', lambda x: f'{x:.4f}')

TRAIN_PATH = "./Database/train.csv"
TEST_PATH = "./Database/test.csv"
PREDICTIONS_OUTPUT_PATH = "./Database/test_predictions.csv"
TARGET_COL = "Skill_Retention_Score"
ID_COL = "Student_ID"
RANDOM_STATE = 42

print("=" * 80)
print("  ANÁLISE EXPLORATÓRIA DE DADOS (EDA) & MODELAGEM DE BASELINE")
print("  Dataset: AI Student Impact Dataset")
print("=" * 80)


  ANÁLISE EXPLORATÓRIA DE DADOS (EDA) & MODELAGEM DE BASELINE
  Dataset: AI Student Impact Dataset


# 2. Carga e Inspeção Inicial dos Dados


In [16]:
if not os.path.exists(TRAIN_PATH):
    raise FileNotFoundError(f"Arquivo de treino não encontrado em {TRAIN_PATH}")

df_train = pd.read_csv(TRAIN_PATH)
print(f"\n[INFO] Dados de treino carregados com sucesso de: {TRAIN_PATH}")
print(f"       Dimensões: {df_train.shape[0]:,} linhas x {df_train.shape[1]} colunas.")

# Identificação de colunas e tipos
print("\n" + "-" * 80)
print("2.1. ESTRUTURA GERAL DAS COLUNAS E TIPOS DE DADOS")
print("-" * 80)
col_info = pd.DataFrame({
    'Tipo': df_train.dtypes,
    'Não-Nulos': df_train.notnull().sum(),
    'Nulos': df_train.isnull().sum(),
    '% Nulos': (df_train.isnull().sum() / len(df_train)) * 100,
    'Uniques': df_train.nunique()
})
print(col_info)



[INFO] Dados de treino carregados com sucesso de: ./Database/train.csv
       Dimensões: 35,000 linhas x 16 colunas.

--------------------------------------------------------------------------------
2.1. ESTRUTURA GERAL DAS COLUNAS E TIPOS DE DADOS
--------------------------------------------------------------------------------
                               Tipo  Não-Nulos  Nulos  % Nulos  Uniques
Student_ID                    int64      35000      0   0.0000    35000
Major_Category                  str      35000      0   0.0000        5
Year_of_Study                   str      35000      0   0.0000        5
Pre_Semester_GPA            float64      35000      0   0.0000     2316
Weekly_GenAI_Hours          float64      35000      0   0.0000     3361
Primary_Use_Case                str      35000      0   0.0000        5
Prompt_Engineering_Skill        str      35000      0   0.0000        3
Tool_Diversity                int64      35000      0   0.0000        5
Paid_Subscription    

# 3. Análise Exploratória Detalhada (EDA)


In [ ]:
print("\n" + "-" * 80)
print("3. ANÁLISE EXPLORATÓRIA DOS DADOS (EDA)")
print("-" * 80)

print("\n>>> 3.1. Análise da Variável Alvo (Target):", TARGET_COL)
target_stats = df_train[TARGET_COL].describe()
print(target_stats)
target_skew = df_train[TARGET_COL].skew()
target_kurt = df_train[TARGET_COL].kurtosis()
print(f"Assimetria (Skewness): {target_skew:.4f} (Distribuição aproximadamente simétrica/levemente alongada)")
print(f"Curtose (Kurtosis):     {target_kurt:.4f}")


--------------------------------------------------------------------------------
3. ANÁLISE EXPLORATÓRIA DOS DADOS (EDA)
--------------------------------------------------------------------------------

>>> 3.1. Análise da Variável Alvo (Target): Skill_Retention_Score
count   35000.0000
mean       75.8046
std        13.2516
min        10.7800
25%        66.8175
50%        75.9900
75%        85.1100
max       100.0000
Name: Skill_Retention_Score, dtype: float64
Assimetria (Skewness): -0.2166 (Distribuição aproximadamente simétrica/levemente alongada)
Curtose (Kurtosis):     -0.1929


In [ ]:
print("\n>>> 3.2. Cardinalidade e Distribuição das Variáveis Categóricas/Booleanas")
cat_cols = [c for c in df_train.columns if df_train[c].dtype in ['object', 'bool', 'string'] and c != TARGET_COL]

for c in cat_cols:
    val_counts = df_train[c].value_counts(normalize=True) * 100
    counts_raw = df_train[c].value_counts()
    dist_df = pd.DataFrame({'Total': counts_raw, 'Percentual (%)': val_counts})
    print(f"\nVariável: [{c}] (Cardinalidade = {df_train[c].nunique()}):")
    print(dist_df.to_string())


>>> 3.2. Cardinalidade e Distribuição das Variáveis Categóricas/Booleanas

Variável: [Major_Category] (Cardinalidade = 5):
                Total  Percentual (%)
Major_Category                       
STEM            10508         30.0229
Business         8753         25.0086
Humanities       7024         20.0686
Medical          4511         12.8886
Arts             4204         12.0114

Variável: [Year_of_Study] (Cardinalidade = 5):
               Total  Percentual (%)
Year_of_Study                       
Junior          7733         22.0943
Freshman        7700         22.0000
Senior          7431         21.2314
Sophomore       6932         19.8057
Graduate        5204         14.8686

Variável: [Primary_Use_Case] (Cardinalidade = 5):
                           Total  Percentual (%)
Primary_Use_Case                                
Debugging/Troubleshooting   8581         24.5171
Copywriting/Drafting        8443         24.1229
Ideation                    7497         21.4200
Summari

In [ ]:
print("\n>>> 3.3. Estatísticas Descritivas das Variáveis Numéricas (Features)")
num_cols = [c for c in df_train.select_dtypes(include=[np.number]).columns if c not in [ID_COL, TARGET_COL]]
print(df_train[num_cols].describe().T)


>>> 3.3. Estatísticas Descritivas das Variáveis Numéricas (Features)
                                count    mean    std    min    25%     50%     75%     max
Pre_Semester_GPA           35000.0000  3.1469 0.4791 1.1830 2.8360  3.2110  3.5210  3.9980
Weekly_GenAI_Hours         35000.0000  8.4366 8.2825 0.0000 2.3900  5.8000 11.6925 40.0000
Tool_Diversity             35000.0000  2.8027 1.1912 1.0000 2.0000  3.0000  4.0000  5.0000
Traditional_Study_Hours    35000.0000 11.1957 5.1529 1.0000 7.5200 11.1800 14.7100 35.8600
Perceived_AI_Dependency    35000.0000  3.5019 1.8214 1.0000 2.0000  3.0000  5.0000 10.0000
Anxiety_Level_During_Exams 35000.0000  4.2768 2.1491 1.0000 3.0000  4.0000  6.0000 10.0000
Post_Semester_GPA          35000.0000  3.3496 0.4965 1.0000 3.0230  3.4240  3.7490  4.0000

>>> 3.4. Matriz de Correlação Linear de Pearson com o Target:
                            Correlação com Target
Skill_Retention_Score                      1.0000
Tool_Diversity                         

In [ ]:
num_cols = [c for c in df_train.select_dtypes(include=[np.number]).columns if c not in [ID_COL, TARGET_COL]]

print("\n>>> 3.4. Matriz de Correlação Linear de Pearson com o Target:")
corr_with_target = df_train[num_cols + [TARGET_COL]].corr()[TARGET_COL].sort_values(ascending=False)
print(corr_with_target.to_frame(name="Correlação com Target"))

In [ ]:
print("\n>>> 3.5. Média e Desvio-Padrão do Target por Categoria Chave:")

for c in ['Prompt_Engineering_Skill', 'Burnout_Risk_Level', 'Institutional_Policy', 'Primary_Use_Case']:
    grouped = df_train.groupby(c)[TARGET_COL].agg(['count', 'mean', 'std']).sort_values('mean', ascending=False)
    print(f"\nTarget por [{c}]:")
    print(grouped)


>>> 3.5. Média e Desvio-Padrão do Target por Categoria Chave:

Target por [Prompt_Engineering_Skill]:
                          count    mean     std
Prompt_Engineering_Skill                       
Advanced                   9647 82.0681 12.4968
Intermediate              12428 75.7656 12.3689
Beginner                  12925 71.1672 12.6922

Target por [Burnout_Risk_Level]:
                    count    mean     std
Burnout_Risk_Level                       
Low                 11473 76.4818 12.8123
Medium              14830 76.1647 12.9848
High                 8697 74.2972 14.1276

Target por [Institutional_Policy]:
                       count    mean     std
Institutional_Policy                        
Actively_Encouraged    10473 76.1462 13.0706
Allowed_With_Citation  17658 75.9322 13.2333
Strict_Ban              6869 74.9558 13.5361

Target por [Primary_Use_Case]:
                           count    mean     std
Primary_Use_Case                                
Debugging/Troubleshoot

# 4. Considerações Metodológicas: Ideal vs. Baseline


### PROCEDIMENTOS IDEAIS PARA AMBIENTE DE PRODUÇÃO / SOLUÇÃO DEFINITIVA:

1. **Feature Engineering Avançada**:
   - Razão de horas de estudo: Weekly_GenAI_Hours / (Traditional_Study_Hours + 1).

   - Variação do GPA: Delta_GPA = Post_Semester_GPA - Pre_Semester_GPA.

   - Índice de Dependência x Eficiência: (Perceived_AI_Dependency * Weekly_GenAI_Hours).

2. **Tratamento Específico de Cardinalidade e Ordinalidade**:
   - Ordinal Encoding ordenado logicamente para:
     * Year_of_Study (Freshman < Sophomore < Junior < Senior < Graduate).
     * Prompt_Engineering_Skill (Beginner < Intermediate < Advanced).
     * Burnout_Risk_Level (Low < Medium < High).

   - Target Encoding ou One-Hot com regularização para categorias nominais (Major_Category, Primary_Use_Case).

3. **Normalização e Robustez**:
   - RobustScaler ou QuantileTransformer para variáveis assimétricas e com outliers.

4. **Estratégia de Validação**:
   - K-Fold Cross-Validation Estratificado por quantis do Target (10 Folds).

5. **Algoritmos Avançados**:
   - Gradient Boosting (LightGBM, XGBoost, CatBoost) com busca Bayesiana de hiperparâmetros.

### PROCEDIMENTOS APLICADOS NO BASELINE (SIMPLES, ROBUSTO E REPRODUTÍVEL):

1. Exclusão do Student_ID (evita data leakage e memorização espúria de ID).

2. Imputação rápida de valores ausentes (Mediana para numéricos, Moda para categóricos).

3. One-Hot Encoding para todas as variáveis categóricas (lidando com novas categorias via ignore).

4. Divisão Holdout simples de 80% treino e 20% validação (random_state fixo).

5. Treinamento comparativo de 4 modelos de baseline com diferentes níveis de complexidade:
   - Baseline 0: DummyRegressor (Previsão da média - Benchmark zero-skill)

   - Baseline 1: Regressão Linear (OLS - Relações lineares diretas)

   - Baseline 2: Ridge Regression (Regressão Linear com Regularização L2)

   - Baseline 3: Random Forest Regressor (Ensemble não-linear com árvores de decisão)


# 5. Pré-processamento e Divisão de Dados


In [21]:
print("-" * 80)
print("5. PRÉ-PROCESSAMENTO E DIVISÃO TREINO / VALIDAÇÃO (80 / 20)")
print("-" * 80)

# Separação de X e y
features = [c for c in df_train.columns if c not in [ID_COL, TARGET_COL]]
X = df_train[features].copy()
y = df_train[TARGET_COL].copy()

# Identificação dos tipos de colunas para o pipeline
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object', 'bool', 'string']).columns.tolist()

print(f"[Features Numéricas]   ({len(numeric_features)}): {numeric_features}")
print(f"[Features Categóricas] ({len(categorical_features)}): {categorical_features}")

# Pipelines de pré-processamento via ColumnTransformer
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Holdout 80% Treino e 20% Validação
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print(f"\nDivisão realizada com sucesso:")
print(f"  Treino:     {X_train.shape[0]:,} amostras ({(len(X_train)/len(X))*100:.1f}%)")
print(f"  Validação:  {X_val.shape[0]:,} amostras ({(len(X_val)/len(X))*100:.1f}%)")


--------------------------------------------------------------------------------
5. PRÉ-PROCESSAMENTO E DIVISÃO TREINO / VALIDAÇÃO (80 / 20)
--------------------------------------------------------------------------------
[Features Numéricas]   (7): ['Pre_Semester_GPA', 'Weekly_GenAI_Hours', 'Tool_Diversity', 'Traditional_Study_Hours', 'Perceived_AI_Dependency', 'Anxiety_Level_During_Exams', 'Post_Semester_GPA']
[Features Categóricas] (7): ['Major_Category', 'Year_of_Study', 'Primary_Use_Case', 'Prompt_Engineering_Skill', 'Paid_Subscription', 'Institutional_Policy', 'Burnout_Risk_Level']

Divisão realizada com sucesso:
  Treino:     28,000 amostras (80.0%)
  Validação:  7,000 amostras (20.0%)


# 6. Treinamento e Avaliação dos Modelos de Baseline


In [22]:
print("\n" + "=" * 80)
print("6. TREINAMENTO DOS MODELOS DE BASELINE")
print("=" * 80)

# Dicionário de modelos a avaliar
models = {
    "DummyRegressor (Média)": DummyRegressor(strategy="mean"),
    "Regressão Linear (OLS)": LinearRegression(),
    "Ridge Regression (L2)": Ridge(alpha=1.0, random_state=RANDOM_STATE),
    "Random Forest Regressor": RandomForestRegressor(
        n_estimators=100, max_depth=12, random_state=RANDOM_STATE, n_jobs=-1
    )
}

results_list = []
trained_pipelines = {}

for name, model in models.items():
    print(f"\n--> Treinando [{name}]...")
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])
    
    # Ajuste no conjunto de treino
    pipe.fit(X_train, y_train)
    trained_pipelines[name] = pipe
    
    # Avaliação no treino e na validação
    y_pred_train = pipe.predict(X_train)
    y_pred_val = pipe.predict(X_val)
    
    # Cálculo das métricas de validação
    rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
    mae_val = mean_absolute_error(y_val, y_pred_val)
    r2_val = r2_score(y_val, y_pred_val)
    mape_val = mean_absolute_percentage_error(y_val, y_pred_val) * 100
    
    # Métricas de treino para checar sobreajuste (overfitting)
    rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
    r2_train = r2_score(y_train, y_pred_train)
    
    results_list.append({
        "Modelo": name,
        "RMSE (Val)": rmse_val,
        "MAE (Val)": mae_val,
        "R² (Val)": r2_val,
        "MAPE (%)": mape_val,
        "RMSE (Treino)": rmse_train,
        "R² (Treino)": r2_train
    })



6. TREINAMENTO DOS MODELOS DE BASELINE

--> Treinando [DummyRegressor (Média)]...

--> Treinando [Regressão Linear (OLS)]...

--> Treinando [Ridge Regression (L2)]...

--> Treinando [Random Forest Regressor]...


# 7. Ranking dos Modelos e Seleção do Campeão


In [23]:
print("\n" + "=" * 80)
print("7. RANKING E COMPARATIVO DE PERFORMANCE NA VALIDAÇÃO")
print("=" * 80)

df_results = pd.DataFrame(results_list)
# Ordenar por menor RMSE na validação (e maior R²)
df_results = df_results.sort_values(by="RMSE (Val)", ascending=True).reset_index(drop=True)
df_results.index = df_results.index + 1
df_results.index.name = "Posição"

print(df_results.to_string())

# Seleção do melhor modelo
best_model_name = df_results.iloc[0]["Modelo"]
best_model_rmse = df_results.iloc[0]["RMSE (Val)"]
best_model_r2 = df_results.iloc[0]["R² (Val)"]
best_pipeline = trained_pipelines[best_model_name]

print("\n" + "*" * 80)
print(f"  MODELO VENCEDOR SELECIONADO: [{best_model_name}]")
print(f"  RMSE de Validação: {best_model_rmse:.4f}")
print(f"  R² de Validação:   {best_model_r2:.4f}")
print("*" * 80)



7. RANKING E COMPARATIVO DE PERFORMANCE NA VALIDAÇÃO
                          Modelo  RMSE (Val)  MAE (Val)  R² (Val)  MAPE (%)  RMSE (Treino)  R² (Treino)
Posição                                                                                                
1        Random Forest Regressor     11.7807     9.4767    0.1955   13.4633         9.9045       0.4438
2          Ridge Regression (L2)     11.9822     9.6632    0.1678   13.7634        12.0170       0.1812
3         Regressão Linear (OLS)     11.9822     9.6632    0.1678   13.7634        12.0170       0.1812
4         DummyRegressor (Média)     13.1353    10.5245   -0.0001   15.0829        13.2803       0.0000

********************************************************************************
  MODELO VENCEDOR SELECIONADO: [Random Forest Regressor]
  RMSE de Validação: 11.7807
  R² de Validação:   0.1955
********************************************************************************


# 8. Inferência Única no Conjunto de Teste (./Database/test.csv)


In [24]:
print("\n" + "=" * 80)
print("8. EXECUÇÃO DA INFERÊNCIA NO CONJUNTO DE TESTE")
print("=" * 80)

if not os.path.exists(TEST_PATH):
    raise FileNotFoundError(f"Arquivo de teste não encontrado em {TEST_PATH}")

df_test = pd.read_csv(TEST_PATH)
print(f"[INFO] Dataset de teste carregado de: {TEST_PATH}")
print(f"       Dimensões: {df_test.shape[0]:,} linhas x {df_test.shape[1]} colunas.")

# Verificar ausência do target no teste
has_target_in_test = TARGET_COL in df_test.columns
print(f"       Presença da coluna Target '{TARGET_COL}' no teste: {has_target_in_test}")

# Garantir mesmas colunas de entrada
X_test = df_test[features].copy()

# Inferência ÚNICA com o melhor modelo
print(f"\n[INFERÊNCIA] Executando predições com o modelo vencedor [{best_model_name}]...")
test_predictions = best_pipeline.predict(X_test)

# Criação do DataFrame de submissão/predições
df_predictions = pd.DataFrame({
    ID_COL: df_test[ID_COL],
    f"{TARGET_COL}_Pred": np.clip(test_predictions, 0.0, 100.0) # Pontuação no intervalo válido [0, 100]
})

# Resumo Estatístico das Predições
print("\n" + "-" * 80)
print("ESTATÍSTICAS DESCRITIVAS DAS PREDIÇÕES GERADAS NO CONJUNTO DE TESTE:")
print("-" * 80)
print(df_predictions[f"{TARGET_COL}_Pred"].describe().to_frame(name="Estatísticas Predição").to_string())

# Salvar predições em arquivo CSV
df_predictions.to_csv(PREDICTIONS_OUTPUT_PATH, index=False)
print(f"\n[SUCESSO] Predições salvas em: {PREDICTIONS_OUTPUT_PATH}")

# Amostra das predições
print("\nPrimeiras 10 linhas das predições geradas:")
print(df_predictions.head(10).to_string(index=False))

print("\n" + "=" * 80)
print("EXECUÇÃO CONCLUÍDA COM SUCESSO!")
print("=" * 80)



8. EXECUÇÃO DA INFERÊNCIA NO CONJUNTO DE TESTE
[INFO] Dataset de teste carregado de: ./Database/test.csv
       Dimensões: 15,000 linhas x 15 colunas.
       Presença da coluna Target 'Skill_Retention_Score' no teste: False

[INFERÊNCIA] Executando predições com o modelo vencedor [Random Forest Regressor]...

--------------------------------------------------------------------------------
ESTATÍSTICAS DESCRITIVAS DAS PREDIÇÕES GERADAS NO CONJUNTO DE TESTE:
--------------------------------------------------------------------------------
       Estatísticas Predição
count             15000.0000
mean                 75.9029
std                   6.2569
min                  33.4545
25%                  71.9779
50%                  75.7153
75%                  79.3635
max                  94.5194

[SUCESSO] Predições salvas em: ./Database/test_predictions.csv

Primeiras 10 linhas das predições geradas:
 Student_ID  Skill_Retention_Score_Pred
     133554                     73.5997
     109